In [ ]:
import os

os.environ["PYKEOPS_VERBOSE"] = "0"
os.environ["KEOPS_VERBOSE"] = "0"

In [ ]:
import torch

# Force CUDA re-initialisation
if torch.cuda.is_available():
    torch.cuda.init()

print(torch.cuda.device_count())  # should now be > 0

In [ ]:
import sys
import os
import pickle
import logging
import warnings
import re
from typing import Dict, List
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.gridspec import GridSpec, GridSpecFromSubplotSpec
import geopandas as gpd

import torch

import massbalancemachine as mbm

sys.path.append(os.path.join(os.getcwd(), '../../'))
from regions.TF_Europe.scripts.config_TF_Europe import *
from regions.TF_Europe.scripts.config_models import *
from regions.TF_Europe.scripts.dataset import *
from regions.TF_Europe.scripts.plotting import *
from regions.TF_Europe.scripts.models import *

warnings.filterwarnings('ignore')
%load_ext autoreload
%autoreload 2

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(message)s')

cfg = mbm.EuropeTFConfig()
mbm.utils.seed_all(cfg.seed)
mbm.utils.free_up_cuda()
mbm.plots.use_mbm_style()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(torch.cuda.is_available())
print(torch.cuda.device_count())

if torch.cuda.device_count() == 0:
    print("No GPU detected. Using CPU.")
    device = torch.device("cpu")

In [ ]:
BASE_DIR = Path(cfg.dataPath) / path_cache / f"WITHIN_REGION"
print(f"Base directory for data: {BASE_DIR}")

# make directories if they don't exist
os.makedirs(BASE_DIR, exist_ok=True)

## Within region modelling:

### Read stakes datasets:

In [ ]:
"""
Examples of loading data:
# Load Switzerland only
df = load_stakes(cfg, "CH")

# Load all Central Europe (SJM+CH+IT+AT when you add them)
df_ceu = load_stakes_for_rgi_region(cfg, "11")

# Load all Europe regions configured
dfs = {rid: load_stakes_for_rgi_region(cfg, rid) for rid in RGI_REGIONS.keys()}"""

# Load all Europe regions configured
dfs = {rid: load_stakes_for_rgi_region(cfg, rid) for rid in RGI_REGIONS.keys()}

rgi_regions = dfs.keys()
num_measurements = {}
for rgi_region in rgi_regions:
    src_code_regions = dfs[rgi_region].SOURCE_CODE.unique()
    for src_code in src_code_regions:
        num_measurements[src_code] = len(
            dfs[rgi_region][dfs[rgi_region].SOURCE_CODE == src_code])
num_measurements

In [ ]:
# run it
summarize_and_plot_all_regions(dfs)

In [ ]:
# run it
plot_mb_distributions_all_regions(dfs)

## Datasets:

In [ ]:
WITHIN_CODES = ["CH", "NOR", "ISL", "FR", "IT_AT", "SJM", "CENTRALASIA", "ALA"]

run_flag_by_code = {code: False for code in WITHIN_CODES}

monthly_cache = build_monthly_cache(
    cfg=cfg,
    dfs=dfs,
    paths_multi=paths_multi,
    vois_climate=VOIS_CLIMATE,
    vois_topographical=STATIC_COLS,
    region_codes=WITHIN_CODES,
    run_flag_by_code=run_flag_by_code,
)

### Subregions:

In [ ]:
region_colors = [
    "red", "blue", "green", "purple", "orange", "darkred", "cadetblue",
    "darkgreen", "darkpurple", "pink", "gray", "black"
]

# Central Asia
dfs, glacier_dict_ca = add_o2region_to_dfs(dfs, '13', cfg)

# Map O2Region into monthly_cache for CENTRALASIA
monthly_cache['CENTRALASIA']['data_monthly']['O2Region'] = (
    monthly_cache['CENTRALASIA']['data_monthly']['GLACIER'].map({
        k:
        v['O2Region']
        for k, v in glacier_dict_ca.items()
    }))
monthly_cache['CENTRALASIA']['data_monthly_aug']['O2Region'] = (
    monthly_cache['CENTRALASIA']['data_monthly_aug']['GLACIER'].map({
        k:
        v['O2Region']
        for k, v in glacier_dict_ca.items()
    }))

# Central Asia subregions (O2Region already mapped onto dfs['13'])
monthly_cache = split_monthly_cache_by_glaciers(
    monthly_cache,
    'CENTRALASIA',
    subregions={
        'CA_12': {
            'o2region_col': 'O2Region',
            'values': ['1', '2']
        },
        'CA_3': {
            'o2region_col': 'O2Region',
            'values': ['3']
        },
        'CA_4': {
            'o2region_col': 'O2Region',
            'values': ['4']
        },
    },
)

o2_color_map = {
    r: region_colors[i % len(region_colors)]
    for i, r in enumerate(
        sorted(v['O2Region'] for v in glacier_dict_ca.values()
               if v['O2Region'] is not None))
}
color_map = {
    g: o2_color_map[info['O2Region']]
    for g, info in glacier_dict_ca.items() if info['O2Region'] is not None
}
m = plot_stakes_folium(dfs['13'], color_map=color_map, tooltip_col='O2Region')
m

In [ ]:
# Alaska
dfs, glacier_dict_alaska = add_o2region_to_dfs(dfs,
                                               '01',
                                               cfg,
                                               deduplicate_glaciers=True)

# Map O2Region into monthly_cache for ALA
monthly_cache['ALA']['data_monthly']['O2Region'] = (
    monthly_cache['ALA']['data_monthly']['GLACIER'].map({
        k: v['O2Region']
        for k, v in glacier_dict_alaska.items()
    }))
monthly_cache['ALA']['data_monthly_aug']['O2Region'] = (
    monthly_cache['ALA']['data_monthly_aug']['GLACIER'].map({
        k: v['O2Region']
        for k, v in glacier_dict_alaska.items()
    }))

# Alaska subregions (O2Region already mapped onto dfs['01'])
monthly_cache = split_monthly_cache_by_glaciers(
    monthly_cache,
    'ALA',
    subregions={
        'ALA_2': {
            'o2region_col': 'O2Region',
            'values': ['2']
        },
        'ALA_4': {
            'o2region_col': 'O2Region',
            'values': ['4']
        },
        'ALA_6': {
            'o2region_col': 'O2Region',
            'values': ['6']
        },
    },
)

o2_color_map = {
    r: region_colors[i % len(region_colors)]
    for i, r in enumerate(
        sorted(v['O2Region'] for v in glacier_dict_alaska.values()
               if v['O2Region'] is not None))
}
color_map = {
    g: o2_color_map[info['O2Region']]
    for g, info in glacier_dict_alaska.items() if info['O2Region'] is not None
}

m = plot_stakes_folium(dfs['01'], color_map=color_map, tooltip_col='O2Region')
m

In [ ]:
# IT/AT split
monthly_cache = split_monthly_cache_by_glaciers(
    monthly_cache,
    'IT_AT',
    subregions={
        'IT': IT_GLACIERS,
        'AT': AT_GLACIERS
    },
    drop_parent=True,
)

print(monthly_cache['AT']['data_monthly'].ID.nunique(),
monthly_cache['IT']['data_monthly'].ID.nunique())

In [ ]:
# print number of glaciers per subregion:
for region in monthly_cache.keys():
    print("{}: {} glaciers".format(region, len(monthly_cache[region]['data_monthly'].GLACIER.unique())))

In [ ]:
monthly_cache['ALA']['data_monthly'].GLACIER.unique()

### Split into train/test:

In [ ]:
def prepare_within_region_pairs_from_cache(
    cfg,
    monthly_cache,
    within_codes,
    gl_level_split=None,
    holdout_frac=0.20,
    n_restarts=100,
    min_train_glaciers=3,
    checkpoint_path=None,
):
    gl_level_split = gl_level_split or []

    res_for_scalers = {
        code: {
            "df_train": monthly_cache[code]["data_monthly"]
        }
        for code in within_codes
    }
    scaler_m, scaler_s, _ = build_global_scalers_multi_source_simple(
        res_for_scalers,
        monthly_cols=MONTHLY_COLS,
        static_cols=STATIC_COLS,
    )
    _, _, blur_joint = estimate_global_bandwidths_simple(
        res_for_scalers,
        monthly_cols=MONTHLY_COLS,
        static_cols=STATIC_COLS,
        scaler_m=scaler_m,
        scaler_s=scaler_s,
        seed=cfg.seed,
    )
    print(f"Estimated blur_joint={blur_joint:.4f}")

    # ── load checkpoint if it exists ─────────────────────────────────────────
    res_by_code = {}
    split_info_by_code = {}
    failed_codes = []

    if checkpoint_path is not None and Path(checkpoint_path).exists():
        with open(checkpoint_path, "rb") as f:
            ckpt = pickle.load(f)
        res_by_code = ckpt.get("res_by_code", {})
        split_info_by_code = ckpt.get("split_info_by_code", {})
        failed_codes = ckpt.get("failed_codes", [])
        already_done = set(res_by_code.keys()) | set(failed_codes)
        print(f"  Resuming from checkpoint: {len(res_by_code)} done, "
              f"{len(failed_codes)} failed, "
              f"{len(already_done)} skipped")
    else:
        already_done = set()

    for code in within_codes:
        if code in already_done:
            print(f"  Skipping {code} (already in checkpoint)")
            continue

        print(f"\n{'='*50}\nWithin-region: {code}")

        try:
            if code not in monthly_cache:
                raise KeyError(f"Code not in monthly_cache: {code}")

            data_monthly = monthly_cache[code]["data_monthly"].copy()
            data_monthly_aug = monthly_cache[code]["data_monthly_aug"].copy()
            head_pad = monthly_cache[code]["months_head_pad"]
            tail_pad = monthly_cache[code]["months_tail_pad"]

            use_gl_split = code in gl_level_split

            # guard: fall back to ID-split if too few glaciers
            if use_gl_split:
                n_glaciers = data_monthly["GLACIER"].nunique()
                min_total = math.ceil(min_train_glaciers / (1 - holdout_frac))
                if n_glaciers < min_total:
                    print(f"  ⚠ Only {n_glaciers} glacier(s) for {code} "
                          f"(need ≥{min_total}) — falling back to ID-split")
                    use_gl_split = False

            split_unit = "GLACIER" if use_gl_split else "ID"

            if use_gl_split:
                best_split = split_train_test_sinkhorn(
                    df_region=data_monthly,
                    monthly_cols=MONTHLY_COLS,
                    static_cols=STATIC_COLS,
                    scaler_m=scaler_m,
                    scaler_s=scaler_s,
                    blur_joint=blur_joint,
                    train_frac=1 - holdout_frac,
                    n_restarts=n_restarts,
                    frac_tol=0.05,
                    seed=cfg.seed,
                )
                best_score = best_split["best_score"]
            else:
                print(f"  Using ID-level split")
                best_split = split_pool_holdout_by_id(
                    df_region=data_monthly,
                    id_col="ID",
                    train_frac=1 - holdout_frac,
                    seed=cfg.seed,
                )
                best_score = float("nan")

            test_units = sorted(best_split["test_glaciers"])
            train_units = sorted(best_split["train_glaciers"])

            print(f"  Split on      : {split_unit}")
            print(f"  Train units   : {len(train_units)}")
            print(f"  Test  units   : {len(test_units)}")
            print(f"  Train frac    : {best_split['actual_train_frac']:.1%}")

            def _make_split(data, test_units, split_on):
                data = data.reset_index(drop=True)
                dataloader = mbm.dataloader.DataLoader(
                    cfg,
                    data=data,
                    random_seed=cfg.seed,
                    meta_data_columns=cfg.metaData,
                )
                _, test_set, train_set = get_CV_splits(
                    dataloader,
                    test_split_on=split_on,  # was: split_on=  ← wrong kwarg name
                    test_splits=
                    test_units,  # was: test_glaciers=  ← wrong kwarg name
                    random_state=cfg.seed,
                )
                # Reconstruct full DataFrames (get_CV_splits returns dicts, not raw DFs)
                train_df = train_set["df_X"].copy()
                train_df["POINT_BALANCE"] = train_set["y"]
                test_df = test_set["df_X"].copy()
                test_df["POINT_BALANCE"] = test_set["y"]
                return train_df, test_df

            train_set, test_set = _make_split(data_monthly, test_units,
                                              split_unit)
            train_set_aug, test_set_aug = _make_split(data_monthly_aug,
                                                      test_units, split_unit)

            res_by_code[code] = {
                "df_train": train_set,
                "df_test": test_set,
                "df_train_aug": train_set_aug,
                "df_test_aug": test_set_aug,
                "months_head_pad": head_pad,  # renamed from "head_pad"
                "months_tail_pad": tail_pad,  # renamed from "tail_pad"
                "scaler_m": scaler_m,
                "scaler_s": scaler_s,
                "blur_joint": blur_joint,
            }
            split_info_by_code[code] = {
                "train_glaciers": train_units,
                "test_glaciers": test_units,
                "actual_train_frac": best_split["actual_train_frac"],
                "best_score": best_score,
                "split_unit": split_unit,
            }

        except Exception as e:
            print(f"  ✗ Failed for {code}: {e}")
            failed_codes.append(code)

        finally:
            if checkpoint_path is not None:
                with open(checkpoint_path, "wb") as f:
                    pickle.dump(
                        {
                            "res_by_code": res_by_code,
                            "split_info_by_code": split_info_by_code,
                            "failed_codes": failed_codes,
                        }, f)

    if failed_codes:
        print(f"\n⚠ Failed regions: {failed_codes}")

    return res_by_code, split_info_by_code

In [ ]:
RECOMPUTE_SPLITS = False
if RECOMPUTE_SPLITS:
    for p in [
            BASE_DIR / "within_region_splits_checkpoint_gl.pkl",
            BASE_DIR / "within_region_splits_checkpoint_id.pkl",
    ]:
        if p.exists():
            os.remove(p)
            print(f"Deleted {p}")

# Update WITHIN_CODES to include subregions after all splits
WITHIN_CODES_ALL = [
    c for c in WITHIN_CODES if c not in {'IT_AT', 'CENTRALASIA', 'ALA'}
] + ['IT', 'AT', 'CA_12', 'CA_3', 'CA_4', 'ALA_2', 'ALA_4', 'ALA_6']

SPLITS_CACHE = BASE_DIR / "within_region_splits_cache_bis.pkl"

# Regions where we prefer glacier-level split (function will auto-fall back
# to ID-split if a region has too few glaciers)
GL_LEVEL_SPLIT = [
    'CH', 'NOR', 'ISL', 'FR', 'IT', 'AT', 'CA_12', 'CA_3', 'CA_4', 'ALA_2',
    'ALA_4', 'ALA_6'
]
# Regions where we always want an ID-level split (in addition to GL above)
ID_LEVEL_SPLIT = ['SJM', 'CA_3', 'CA_4', 'ALA_2', 'ALA_4', 'ALA_6']

if RECOMPUTE_SPLITS or not SPLITS_CACHE.exists():
    res_within_by_code_gl, split_info_by_code_gl = prepare_within_region_pairs_from_cache(
        cfg=cfg,
        monthly_cache=monthly_cache,
        within_codes=WITHIN_CODES_ALL,
        gl_level_split=GL_LEVEL_SPLIT,
        checkpoint_path=BASE_DIR / "within_region_splits_checkpoint_gl.pkl",
    )

    res_within_by_code_id, split_info_by_code_id = prepare_within_region_pairs_from_cache(
        cfg=cfg,
        monthly_cache=monthly_cache,
        within_codes=ID_LEVEL_SPLIT,
        gl_level_split=[],
        checkpoint_path=BASE_DIR / "within_region_splits_checkpoint_id.pkl",
    )

    with open(SPLITS_CACHE, "wb") as f:
        pickle.dump(
            {
                "res_within_by_code_gl": res_within_by_code_gl,
                "split_info_by_code_gl": split_info_by_code_gl,
                "res_within_by_code_id": res_within_by_code_id,
                "split_info_by_code_id": split_info_by_code_id,
            }, f)
    print(f"Saved splits to cache: {SPLITS_CACHE}")
else:
    with open(SPLITS_CACHE, "rb") as f:
        cache = pickle.load(f)
    res_within_by_code_gl = cache["res_within_by_code_gl"]
    split_info_by_code_gl = cache["split_info_by_code_gl"]
    res_within_by_code_id = cache["res_within_by_code_id"]
    split_info_by_code_id = cache["split_info_by_code_id"]
    print(f"Loaded splits from cache: {SPLITS_CACHE}")

# Convenience alias: glacier-split is the primary one used downstream
res_within_by_code = res_within_by_code_gl
split_info_by_code = split_info_by_code_gl

#### Feature overlap:

In [ ]:
# Example usage:
# res_within_by_code is what you got from prepare_monthlies_for_all_regions(...)
figs = plot_overlap_for_all_results(
    results_dict=res_within_by_code,
    cfg=cfg,
    STATIC_COLS=STATIC_COLS,
    MONTHLY_COLS=MONTHLY_COLS,
    n_iter=1000,
)

In [ ]:
figs_kde = plot_feature_overlap_all_regions(res_within_by_code, STATIC_COLS,
                                            MONTHLY_COLS)

## Model
### Model datasets:

In [ ]:
# Build a combined res_all with disambiguated keys for codes that have BOTH
# a GL-preferred split and a forced ID-level split (CA_3, CA_4, ALA_2, ALA_4, ALA_6).
# SJM only exists in the _id dict, so it passes through unchanged.

overlap_codes = set(res_within_by_code_gl) & set(res_within_by_code_id)

res_within_by_code = {}
for code, val in res_within_by_code_gl.items():
    new_key = f"{code}_GL" if code in overlap_codes else code
    res_within_by_code[new_key] = val

for code, val in res_within_by_code_id.items():
    new_key = f"{code}_ID" if code in overlap_codes else code
    res_within_by_code[new_key] = val

split_info_by_code = {}
for code, val in split_info_by_code_gl.items():
    new_key = f"{code}_GL" if code in overlap_codes else code
    split_info_by_code[new_key] = val

for code, val in split_info_by_code_id.items():
    new_key = f"{code}_ID" if code in overlap_codes else code
    split_info_by_code[new_key] = val

In [ ]:
# Patch old key names from before the head_pad/tail_pad rename, so we don't
# need to re-run the (slow) split computation.
for d in (res_within_by_code, res_within_by_code_gl, res_within_by_code_id):
    for code, val in d.items():
        if "head_pad" in val and "months_head_pad" not in val:
            val["months_head_pad"] = val.pop("head_pad")
        if "tail_pad" in val and "months_tail_pad" not in val:
            val["months_tail_pad"] = val.pop("tail_pad")

logs_cache_dir = BASE_DIR / "datasets"
logs_cache_dir.mkdir(parents=True, exist_ok=True)

lstm_assets = build_or_load_lstm_all(
    cfg=cfg,
    res_all=res_within_by_code,
    MONTHLY_COLS=MONTHLY_COLS,
    STATIC_COLS=STATIC_COLS,
    cache_dir=logs_cache_dir,
    force_recompute=False,
    include_atomic=True,
    include_groups=True,
)

### Parameters:

In [ ]:
# Can be changed to different params per region, but here take default:
def _base_code(key):
    return key[:-3] if key.endswith(("_GL", "_ID")) else key


# LSTM:
lstm_params_by_key = {}
for key in lstm_assets.keys():
    lstm_params_by_key[key] = copy.deepcopy(PARAMS_LSTM)

# Transformer:
transformer_params_by_key = {}
for key in lstm_assets.keys():
    transformer_params_by_key[key] = copy.deepcopy(PARAMS_TRANSFORMER)

### Train model:

In [ ]:
# LSTM
models_lstm, infos_lstm = train_within_region_models_all(
    cfg=cfg,
    lstm_assets_by_key=lstm_assets,
    params_by_key=lstm_params_by_key,
    device=device,
    epochs=N_EPOCHS,
    force_retrain=False,
    models_dir=BASE_DIR / "models",
    prefix="lstm_within",
    date=MODEL_DATE,
    model_type="lstm",
)

# Transformer
# transformer_params_by_key = {code: PARAMS_TRANSFORMER for code in WITHIN_CODES_ALL}
models_tf, infos_tf = train_within_region_models_all(
    cfg=cfg,
    lstm_assets_by_key=lstm_assets,
    params_by_key=transformer_params_by_key,
    device=device,
    epochs=N_EPOCHS,
    force_retrain=False,
    models_dir=BASE_DIR / "models",
    prefix="transformer_within",
    date=MODEL_DATE,
    model_type="transformer",
)

### Evaluate on test:

In [ ]:
def evaluate_all_models(
        cfg,
        models_by_key: dict,
        lstm_assets_by_key: dict,
        device,
        grid_shape=(2, 3),
        grid_figsize=(20, 12),
        ax_xlim=(-16, 9),
        ax_ylim=(-16, 9),
        complement_key=None,
        custom_order=None,
):

    src_label = (complement_key.replace("XREG_", "").replace("_TO_", "")
                 if complement_key else "")

    if custom_order is None:
        keys = sorted(models_by_key.keys())
    else:
        keys = custom_order

    nrows, ncols = grid_shape
    fig_grid, axes = plt.subplots(
        nrows,
        ncols,
        figsize=(12, (90 * nrows) / 25.4),
        sharex=True,
        sharey=True,
    )
    axes = np.array(axes).reshape(-1)
    n_slots = len(axes)

    rows = []
    preds_by_key = {}
    figs_by_key = {}

    for i, key in enumerate(keys):
        domain_shift_key = complement_key + key
        model = models_by_key[domain_shift_key]

        # --- individual fig ---
        metrics, df_preds, fig_ind, ax_ind = evaluate_one_model(
            cfg=cfg,
            model=model,
            device=device,
            lstm_assets_for_key=lstm_assets_by_key[domain_shift_key],
            ax=None,
            ax_xlim=ax_xlim,
            ax_ylim=ax_ylim,
            title=f"{src_label} → {key} – Pred vs Truth (Test)",
            legend_fontsize=NATURE_SPECS["font_max_pt"],
        )

        metrics["key"] = domain_shift_key
        rows.append(metrics)
        preds_by_key[domain_shift_key] = df_preds
        figs_by_key[domain_shift_key] = fig_ind

        plt.close(fig_ind)

        # --- grid subplot ---
        if i < n_slots:
            ax_grid = axes[i]
            evaluate_one_model(
                cfg=cfg,
                model=model,
                device=device,
                lstm_assets_for_key=lstm_assets_by_key[domain_shift_key],
                ax=ax_grid,
                ax_xlim=ax_xlim,
                ax_ylim=ax_ylim,
                title=f"{src_label} → {key}",
                legend_fontsize=NATURE_SPECS["font_max_pt"],
            )
            apply_nature_style(ax_grid,
                               fontsize=NATURE_SPECS["font_max_pt"],
                               box=True)

    # turn off unused axes
    for j in range(len(keys), n_slots):
        axes[j].axis("off")

    for idx, ax in enumerate(axes[:n_slots]):
        row = idx // ncols
        col = idx % ncols
        if row < nrows - 1:
            ax.set_xlabel("")
        if col == 0:
            ax.set_ylabel("Modeled PMB (m w.e.)",
                          fontsize=NATURE_SPECS["font_max_pt"])
        else:
            ax.set_ylabel("")

    for i, ax in enumerate(axes):
        if i != 0:
            leg = ax.get_legend()
            if leg is not None:
                leg.remove()

    # remove all per-axis legends
    for ax in axes[:n_slots]:
        leg = ax.get_legend()
        if leg is not None:
            leg.remove()

    # add a single clean figure-level legend
    legend_handles = [
        mpatches.Patch(color=mbm.plots.COLOR_ANNUAL, label="Annual"),
        mpatches.Patch(color=mbm.plots.COLOR_WINTER, label="Winter"),
    ]
    fig_grid.legend(
        handles=legend_handles,
        loc="lower center",
        bbox_to_anchor=(0.5, -0.02),
        ncols=2,
        frameon=False,
        fontsize=NATURE_SPECS["font_max_pt"],
    )

    fig_grid.suptitle(
        f"Zero-shot transfer performance - source: {src_label}",
        fontsize=NATURE_SPECS["font_max_pt"] + 3,
        y=0.98,
    )
    fig_grid.tight_layout(rect=[0, 0, 1, 0.98])

    df_metrics = pd.DataFrame(rows).set_index("key").sort_index()

    plt.close()

    return df_metrics, preds_by_key, figs_by_key, fig_grid


In [ ]:
df_metrics, preds_by_key, figs_by_key, fig_grid = evaluate_all_models(
    cfg=cfg,
    models_by_key=models_tf,
    lstm_assets_by_key=lstm_assets,
    device=device,
    ax_xlim=(-16, 10),
    ax_ylim=(-16, 10),
    grid_shape=(2, 3),
    grid_figsize=(25, 12),
    complement_key="",
)

display(df_metrics)

In [ ]:
df_metrics_lstm, preds_by_key_lstm, figs_by_key_lstm, fig_grid_lstm = evaluate_all_models(
    cfg=cfg,
    models_by_key=models_lstm,
    lstm_assets_by_key=lstm_assets,
    device=device,
    ax_xlim=(-16, 10),
    ax_ylim=(-16, 10),
    grid_shape=(2, 3),
    grid_figsize=(25, 12),
    complement_key="",
)

display(df_metrics_lstm)

In [ ]:
def plot_within_region_grid(region_to_key, out_path):
    WITHIN_REGIONS = list(region_to_key.keys())
    MAX_COLS = 4
    n_regions = len(WITHIN_REGIONS)
    n_cols = min(MAX_COLS, n_regions)
    n_region_rows = math.ceil(n_regions / n_cols)
    n_rows = 2 * n_region_rows

    # Precompute per-region axis limits
    region_lims_within = {}
    for region in WITHIN_REGIONS:
        key = region_to_key[region]
        vals = lstm_assets[key]["ds_test"].y.cpu().numpy().flatten()
        vals = vals[~np.isnan(vals)]
        pad = (vals.max() - vals.min()) * 0.3
        region_lims_within[region] = (float(vals.min() - pad),
                                      float(vals.max() + pad))

    fig = plt.figure(figsize=(4 * n_cols, 3.8 * 2 * n_region_rows))
    outer_gs = GridSpec(
        n_region_rows,
        1,
        figure=fig,
        hspace=0.25,
        top=0.96,
        bottom=0.04,
        left=0.08,
        right=0.98,
    )
    inner_gs_list = []
    for block_i in range(n_region_rows):
        inner_gs = GridSpecFromSubplotSpec(
            2,
            n_cols,
            subplot_spec=outer_gs[block_i],
            hspace=0.1,
            wspace=0.32,
        )
        inner_gs_list.append(inner_gs)

    axes_grid = []
    for block_i in range(n_region_rows):
        block_axes = []
        for model_i in range(2):
            row_axes = [
                fig.add_subplot(inner_gs_list[block_i][model_i, col])
                for col in range(n_cols)
            ]
            block_axes.append(row_axes)
        axes_grid.append(block_axes)

    for reg_idx, region in enumerate(WITHIN_REGIONS):
        key = region_to_key[region]
        block_i = reg_idx // n_cols
        col = reg_idx % n_cols
        lim = region_lims_within[region]

        for model_i, (model_label, models_by_key) in enumerate([
            ("LSTM", models_lstm),
            ("Transformer", models_tf),
        ]):
            ax = axes_grid[block_i][model_i][col]
            model = models_by_key[key]

            out, test_df_preds, created_fig, ax = evaluate_one_model(
                cfg=cfg,
                model=model,
                device=device,
                lstm_assets_for_key=lstm_assets[key],
                ax=ax,
                ax_xlim=lim,
                ax_ylim=lim,
                legend_fontsize=NATURE_SPECS["font_min_pt"],
            )
            apply_nature_style(ax,
                               fontsize=NATURE_SPECS["font_max_pt"],
                               box=True)
            leg = ax.get_legend()
            if leg:
                leg.remove()

            global_row = block_i * 2 + model_i
            row_letter = chr(ord('a') + global_row)
            panel_label = f"{row_letter}{col + 1}"
            ax.text(0.02,
                    0.98,
                    f"({panel_label})",
                    transform=ax.transAxes,
                    fontsize=NATURE_SPECS["font_max_pt"],
                    va="top",
                    ha="left")

            n_train, n_test = len(
                split_info_by_code[key]['train_glaciers']), len(
                    split_info_by_code[key]['test_glaciers'])

            if model_i == 0:

                def _split_label(key):
                    if key.endswith("_ID"):
                        return "test measurements", "ID split"
                    elif key.endswith("_GL"):
                        return "test glaciers", "GL split"
                    else:
                        # plain key, no suffix — assume GL split (default behavior)
                        return "test glaciers", "GL split"

                unit_label, split_label = _split_label(key)
                ax.set_title("{} ({})".format(
                    REGION_LABELS.get(region, region), 
                    split_label),
                             fontsize=NATURE_SPECS["font_max_pt"] + 1,
                             )
            if col == 0:
                ax.set_ylabel(f"{model_label}\nModeled PMB (m w.e.)",
                              fontsize=NATURE_SPECS["font_max_pt"])
            else:
                ax.set_ylabel("")
            if model_i == 0:
                ax.set_xlabel("")

    for empty_idx in range(n_regions, n_region_rows * n_cols):
        block_i = empty_idx // n_cols
        col = empty_idx % n_cols
        for model_i in range(2):
            axes_grid[block_i][model_i][col].set_visible(False)

    fig.legend(
        handles=[
            mpatches.Patch(color=mbm.plots.COLOR_ANNUAL, label="Annual"),
            mpatches.Patch(color=mbm.plots.COLOR_WINTER, label="Winter"),
        ],
        loc="lower center",
        bbox_to_anchor=(0.5, 0.0),
        ncols=2,
        frameon=False,
        fontsize=NATURE_SPECS["font_max_pt"]+2,
    )
    fig.suptitle(
        f"Within-region performance — LSTM vs Transformer",
        fontsize=NATURE_SPECS["font_max_pt"] + 2,
        y=1.01,
    )
    fig.savefig(out_path, dpi=NATURE_SPECS["dpi"], bbox_inches="tight")
    plt.show()

#### LSTM vs Transformer:

In [ ]:
# --- Variant 2: GL splits for regions that support it, ID splits for SJM/CA/ALA ---
region_to_key_mixed = {
    'CH': 'CH',
    'FR': 'FR',
    'IT': 'IT',
    'AT': 'AT',
    'NOR': 'NOR',
    'ISL': 'ISL',
    'SJM': 'SJM_ID',
    'ALA_2': 'ALA_2_ID',
    'ALA_4': 'ALA_4_ID',
    'ALA_6': 'ALA_6_ID',
    'CA_3': 'CA_3_ID',
    'CA_4': 'CA_4_ID',
}
plot_within_region_grid(
    region_to_key_mixed,
    out_path="figures/paperTF/appendix/app_within_region_lstm_vs_transf_mixed.png",
)

#### Transformer only:

In [ ]:
def plot_within_region_grid_transformer_only(region_to_key, out_path):
    WITHIN_REGIONS = list(region_to_key.keys())
    MAX_COLS = 4
    n_regions = len(WITHIN_REGIONS)
    n_cols = min(MAX_COLS, n_regions)
    n_region_rows = math.ceil(n_regions / n_cols)

    # Precompute per-region axis limits
    region_lims_within = {}
    for region in WITHIN_REGIONS:
        key = region_to_key[region]
        vals = lstm_assets[key]["ds_test"].y.cpu().numpy().flatten()
        vals = vals[~np.isnan(vals)]
        pad = (vals.max() - vals.min()) * 0.3
        region_lims_within[region] = (float(vals.min() - pad),
                                      float(vals.max() + pad))

    fig = plt.figure(figsize=(4 * n_cols, 3.8 * n_region_rows))
    outer_gs = GridSpec(
        n_region_rows, n_cols, figure=fig,
        hspace=0.35, wspace=0.32, top=0.94, bottom=0.10, left=0.08, right=0.98,
    )
    axes_grid = [
        [fig.add_subplot(outer_gs[row, col]) for col in range(n_cols)]
        for row in range(n_region_rows)
    ]

    def _split_label(key):
        if key.endswith("_ID"):
            return "test meas.", "ID split"
        else:
            return "test glaciers", "GL split"

    for reg_idx, region in enumerate(WITHIN_REGIONS):
        key = region_to_key[region]
        row = reg_idx // n_cols
        col = reg_idx % n_cols
        lim = region_lims_within[region]
        ax = axes_grid[row][col]
        model = models_tf[key]

        out, test_df_preds, created_fig, ax = evaluate_one_model(
            cfg=cfg,
            model=model,
            device=device,
            lstm_assets_for_key=lstm_assets[key],
            ax=ax,
            ax_xlim=lim,
            ax_ylim=lim,
            legend_fontsize=NATURE_SPECS["font_min_pt"],
        )
        apply_nature_style(ax, fontsize=NATURE_SPECS["font_max_pt"], box=True)
        leg = ax.get_legend()
        if leg:
            leg.remove()

        panel_label = chr(ord('a') + reg_idx)
        ax.text(0.02, 0.98, f"({panel_label})",
                transform=ax.transAxes,
                fontsize=NATURE_SPECS["font_max_pt"],
                va="top", ha="left")

        n_train, n_test = len(
            split_info_by_code[key]['train_glaciers']), len(
                split_info_by_code[key]['test_glaciers'])
        unit_label, split_label = _split_label(key)

        ax.set_title("{} ({})".format(
            REGION_LABELS.get(region, region), split_label),
                     fontsize=NATURE_SPECS["font_max_pt"] + 1,
                     )
        if col == 0:
            ax.set_ylabel("Modeled PMB (m w.e.)",
                          fontsize=NATURE_SPECS["font_max_pt"])
        else:
            ax.set_ylabel("")

    # hide unused axes
    for empty_idx in range(n_regions, n_region_rows * n_cols):
        row = empty_idx // n_cols
        col = empty_idx % n_cols
        axes_grid[row][col].set_visible(False)

    fig.legend(
    handles=[
            mpatches.Patch(color=mbm.plots.COLOR_ANNUAL, label="Annual"),
            mpatches.Patch(color=mbm.plots.COLOR_WINTER, label="Winter"),
        ],
        loc="lower center", bbox_to_anchor=(0.5, -0.02),
        ncols=2, frameon=False, fontsize=NATURE_SPECS["font_max_pt"],
    )
    fig.suptitle(
        f"Within-region performance — Transformer",
        fontsize=NATURE_SPECS["font_max_pt"] + 2, y=1.01,
    )
    fig.savefig(out_path, dpi=NATURE_SPECS["dpi"], bbox_inches="tight")
    plt.show()

In [ ]:
region_to_key_mixed = {
    'CH': 'CH',
    'FR': 'FR',
    'IT': 'IT',
    'AT': 'AT',
    'NOR': 'NOR',
    'ISL': 'ISL',
    'SJM': 'SJM_ID',
    'ALA_2': 'ALA_2_ID',
    'ALA_4': 'ALA_4_ID',
    'ALA_6': 'ALA_6_ID',
    'CA_3': 'CA_3_GL',
    'CA_4': 'CA_4_ID',
}
# Use whichever region_to_key mapping you prefer (gl, mixed, etc.)
plot_within_region_grid_transformer_only(
    region_to_key_mixed,
    out_path="figures/paperTF/fig_within_region_transformer_only.png",
)

In [ ]:
# Use whichever region_to_key mapping you prefer (gl, mixed, etc.)
region_to_key_gl = {
    'CH': 'CH',
    'FR': 'FR',
    'IT': 'IT',
    'AT': 'AT',
    'NOR': 'NOR',
    'ISL': 'ISL',
    'SJM': 'SJM_GL',
    'ALA_2': 'ALA_2_GL',
    'ALA_4': 'ALA_4_GL',
    'ALA_6': 'ALA_6_GL',
    'CA_3': 'CA_3_GL',
    'CA_4': 'CA_4_GL',
}

plot_within_region_grid_transformer_only(
    region_to_key_gl,
    out_path="figures/paperTF/appendix/fig_within_region_transformer_only_gl_split.png",
)